In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:

df = pd.read_csv(r"D:\Project 8\category_jobs.csv")
df = df.dropna(subset=["clean_title"])
df["published_date"] = pd.to_datetime(df["published_date"], errors="coerce")

In [3]:
# Recommendation base columns
rec_df = df[[
    "title","clean_title","country","job_type","category",
    "avg_hourly_rate","budget","published_date","link"
]].copy().reset_index(drop=True)

In [4]:
# ─── Recency score ─────────────────────────────────────────────────────────
latest_date = rec_df["published_date"].max()
rec_df["recency_days"] = (latest_date - rec_df["published_date"]).dt.days
rec_df["recency_score"] = 1 / (1 + rec_df["recency_days"].fillna(999))

In [5]:
# ─── Pay score (normalised) ───────────────────────────────────────────────
rec_df["pay_raw"] = rec_df["avg_hourly_rate"].fillna(rec_df["budget"].fillna(0) / 40)
max_pay = rec_df["pay_raw"].max()
rec_df["pay_score"] = rec_df["pay_raw"] / max_pay if max_pay > 0 else 0

In [6]:
# ─── TF-IDF vectorizer ────────────────────────────────────────────────────
print("Training TF-IDF vectorizer...")
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=5000)
tfidf_matrix = tfidf.fit_transform(rec_df["clean_title"])
print(f"  Vocabulary size : {len(tfidf.vocabulary_):,}")
print(f"  Matrix shape    : {tfidf_matrix.shape}")

Training TF-IDF vectorizer...
  Vocabulary size : 5,000
  Matrix shape    : (244827, 5000)


In [7]:
# ─── Recommendation functions ────────────────────────────────────────────
def recommend_jobs(user_query, top_n=10, country=None, job_type=None):
    query_vec = tfidf.transform([user_query.lower()])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    temp = rec_df.copy()
    temp["similarity_score"] = sim_scores
    if country:
        temp = temp[temp["country"].str.lower() == country.lower()]
    if job_type:
        temp = temp[temp["job_type"].str.lower() == job_type.lower()]
    return temp.sort_values("similarity_score", ascending=False).head(top_n)

def recommend_jobs_advanced(user_query, top_n=10, country=None, job_type=None, category=None):
    query_vec = tfidf.transform([user_query.lower()])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    temp = rec_df.copy()
    temp["similarity_score"] = sim_scores
    temp["final_score"] = (
        0.70 * temp["similarity_score"] +
        0.20 * temp["recency_score"] +
        0.10 * temp["pay_score"]
    )
    if country:
        temp = temp[temp["country"].str.lower() == country.lower()]
    if job_type:
        temp = temp[temp["job_type"].str.lower() == job_type.lower()]
    if category:
        temp = temp[temp["category"] == category]
    return temp.sort_values("final_score", ascending=False).head(top_n)

In [8]:
# ─── Quick evaluation — precision@5 proxy ────────────────────────────────
test_queries = {
    "data analyst":           "AI_ML_Data",
    "machine learning":       "AI_ML_Data",
    "wordpress developer":    "Web_Development",
    "react frontend":         "Web_Development",
    "seo specialist":         "Marketing_SEO",
    "graphic designer":       "Design_Creative",
    "blockchain smart":       "Blockchain_Web3",
    "flutter mobile app":     "Mobile_Development",
    "virtual assistant":      "Admin_Support",
}

print("\nRecommender evaluation (category precision@5):")
hits = 0
for query, expected_cat in test_queries.items():
    results = recommend_jobs_advanced(query, top_n=5)
    if len(results) > 0:
        top_cat = results["category"].mode().iloc[0]
        match = "✅" if top_cat == expected_cat else "❌"
        if top_cat == expected_cat:
            hits += 1
    else:
        match = "❌ (no results)"
    print(f"  {match}  [{query:<25}]  expected={expected_cat:<20}  got={top_cat}")

precision = hits / len(test_queries)
print(f"\nPrecision@5 (category match): {precision:.0%}  ({hits}/{len(test_queries)})")



Recommender evaluation (category precision@5):
  ✅  [data analyst             ]  expected=AI_ML_Data            got=AI_ML_Data
  ✅  [machine learning         ]  expected=AI_ML_Data            got=AI_ML_Data
  ✅  [wordpress developer      ]  expected=Web_Development       got=Web_Development
  ✅  [react frontend           ]  expected=Web_Development       got=Web_Development
  ✅  [seo specialist           ]  expected=Marketing_SEO         got=Marketing_SEO
  ✅  [graphic designer         ]  expected=Design_Creative       got=Design_Creative
  ❌  [blockchain smart         ]  expected=Blockchain_Web3       got=AI_ML_Data
  ✅  [flutter mobile app       ]  expected=Mobile_Development    got=Mobile_Development
  ✅  [virtual assistant        ]  expected=Admin_Support         got=Admin_Support

Precision@5 (category match): 89%  (8/9)


In [9]:
recommend_jobs("data analyst", top_n=10)
recommend_jobs("machine learning engineer", top_n=10)
recommend_jobs("seo specialist", top_n=10, job_type="Hourly")

,title,clean_title,country,job_type,category,avg_hourly_rate,budget,published_date,link,recency_days,recency_score,pay_raw,pay_score,similarity_score
29384,SEO Specialist,seo specialist,United States,Hourly,Marketing_SEO,15.0,NaN,2024-02-17 07:50:23+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,36,0.027027,15.0,0.00060,1.0
2563,SEO Specialist,seo specialist,United Kingdom,Hourly,Marketing_SEO,NaN,NaN,2024-02-16 17:43:46+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,36,0.027027,0.0,0.00000,1.0
66039,SEO Specialist,seo specialist,India,Hourly,Marketing_SEO,NaN,NaN,2024-02-24 09:05:10+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,29,0.033333,0.0,0.00000,1.0
8686,SEO Specialist,seo specialist,United States,Hourly,Marketing_SEO,22.5,NaN,2024-02-13 22:19:04+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,39,0.025000,22.5,0.00090,1.0
23935,SEO Specialist,seo specialist,Philippines,Hourly,Marketing_SEO,8.5,NaN,2024-02-13 13:03:52+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,40,0.024390,8.5,0.00034,1.0
33618,SEO specialist,seo specialist,Israel,Hourly,Marketing_SEO,10.0,NaN,2024-02-18 09:52:45+00:00,https://www.upwork.com/jobs/SEO-specialist_%7E...,35,0.027778,10.0,0.00040,1.0
94348,SEO Specialist,seo specialist,Turkey,Hourly,Marketing_SEO,15.0,NaN,2024-02-29 13:30:53+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,24,0.040000,15.0,0.00060,1.0
188254,SEO Specialist,seo specialist,United States,Hourly,Marketing_SEO,NaN,NaN,2024-03-11 23:07:33+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,12,0.076923,0.0,0.00000,1.0
163803,SEO Specialist,seo specialist,United States,Hourly,Marketing_SEO,100.0,NaN,2024-03-08 13:56:04+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,16,0.058824,100.0,0.00400,1.0
162042,SEO Specialist,seo specialist,United States,Hourly,Marketing_SEO,NaN,NaN,2024-03-07 17:51:07+00:00,https://www.upwork.com/jobs/SEO-Specialist_%7E...,16,0.058824,0.0,0.00000,1.0


In [10]:
# ─── Save everything ─────────────────────────────────────────────────────
joblib.dump(tfidf,        "tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")
joblib.dump(df, "jobs_data.pkl")
rec_df.to_csv("recommendation_base.csv", index=False)